# Predictive Anayltics: Support Vector Machines

TODO: add embedded, check why there are Nans in lanlong

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [55]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 2000
SPATIAL_UNIT = "community_area" # options: census_tract, h3_cell, community_area
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4h" # options: 1h, 2h, 4h

In [56]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [57]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder
import pytorch_lightning as py_light
print(Hex2VecEmbedder.__mro__)  # should show pytorch_lightning.core.module.LightningModule in the chain

from sklearn.preprocessing import OneHotEncoder

(<class 'srai.embedders.hex2vec.embedder.Hex2VecEmbedder'>, <class 'srai.embedders.count_embedder.CountEmbedder'>, <class 'srai.embedders._base.Embedder'>, <class 'abc.ABC'>, <class 'object'>)


## Preparations

In [58]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [59]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_train.parquet"
DATA_PATH_TEST = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_test.parquet"


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL, # gets encoded
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
    "h3_cell"
]

Load data and select features and target

In [60]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
test = pl.scan_parquet(DATA_PATH_TEST)

In [61]:
print(train.collect_schema().names())

['datetime_hour', 'month', 'weekday', 'hour', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'date', 'is_holiday', 'community_area', 'food_drink', 'landmark', 'shop', 'train_station', 'trip_count', 'trip_seconds_sum', 'trip_seconds_mean', 'trip_seconds_min', 'trip_seconds_max', 'trip_miles_sum', 'trip_miles_mean', 'trip_miles_min', 'trip_miles_max', 'fare_sum', 'fare_mean', 'fare_min', 'fare_max', 'tips_sum', 'tips_mean', 'tips_min', 'tips_max', 'tolls_sum', 'tolls_mean', 'tolls_min', 'tolls_max', 'extras_sum', 'extras_mean', 'extras_min', 'extras_max', 'trip_total_sum', 'trip_total_mean', 'trip_total_min', 'trip_total_max', 'most_common_payment_type']


In [62]:
keep_cols = [
    SPATIAL_UNIT,  # "h3_cell"
    "trip_count",  # target is derived from this
    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    "tmpc", "relh", "sknt", "vsby", "p01m",
    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
    "is_holiday",
    "food_drink", "landmark", "shop", "train_station",
]

train_df = train.select(keep_cols).collect().to_pandas()
test_df = test.select(keep_cols).collect().to_pandas()

In [63]:
train_df.isna().sum()

community_area    0
trip_count        0
month_sin         0
month_cos         0
weekday_sin       0
weekday_cos       0
hour_sin          0
hour_cos          0
tmpc              0
relh              0
sknt              0
vsby              0
p01m              0
skyc1_BKN         0
skyc1_CLR         0
skyc1_FEW         0
skyc1_OVC         0
skyc1_SCT         0
skyc1_VV          0
is_holiday        0
food_drink        0
landmark          0
shop              0
train_station     0
dtype: int64

In [64]:
train_df.count()

community_area    274961
trip_count        274961
month_sin         274961
month_cos         274961
weekday_sin       274961
weekday_cos       274961
hour_sin          274961
hour_cos          274961
tmpc              274961
relh              274961
sknt              274961
vsby              274961
p01m              274961
skyc1_BKN         274961
skyc1_CLR         274961
skyc1_FEW         274961
skyc1_OVC         274961
skyc1_SCT         274961
skyc1_VV          274961
is_holiday        274961
food_drink        274961
landmark          274961
shop              274961
train_station     274961
dtype: int64

In [65]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

test_df["food_drink"] = test_df["food_drink"].fillna(0.0)
test_df["landmark"] = test_df["landmark"].fillna(0.0)
test_df["shop"] = test_df["shop"].fillna(0.0)
test_df["train_station"] = test_df["train_station"].fillna(0.0)


In [66]:
train_df["food_drink"]

0         30.0
1         30.0
2         30.0
3         30.0
4         30.0
          ... 
274956    15.0
274957    15.0
274958    15.0
274959    15.0
274960    15.0
Name: food_drink, Length: 274961, dtype: float64

In [67]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
train_p90 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 90)
train_p70 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 70)
train_p50 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 50)
train_p25 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = 0
train_df.loc[train_df["trip_count"] >= train_p25, "trip_demand"] = 1
train_df.loc[train_df["trip_count"] >= train_p50, "trip_demand"] = 2
train_df.loc[train_df["trip_count"] >= train_p70, "trip_demand"] = 3
train_df.loc[train_df["trip_count"] >= train_p90, "trip_demand"] = 4

test_p90 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 90)
test_p70 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 70)
test_p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
test_p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

test_df["trip_demand"] = 0
test_df.loc[test_df["trip_count"] >= test_p25, "trip_demand"] = 1
test_df.loc[test_df["trip_count"] >= test_p50, "trip_demand"] = 2
test_df.loc[test_df["trip_count"] >= test_p70, "trip_demand"] = 3
test_df.loc[test_df["trip_count"] >= test_p90, "trip_demand"] = 4


In [68]:
print(test_df.loc[test_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[test_df["trip_count"] > 0, "trip_count"].count())

22275
95926


In [69]:
model = SVC()

In [70]:
test_df.loc[test_df["trip_demand"] != "Low", "trip_demand"]

0         1
1         3
2         3
3         3
4         3
         ..
118196    0
118197    0
118198    0
118199    0
118200    0
Name: trip_demand, Length: 118201, dtype: int64

## Encoding

In [71]:
# encoding
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

In [72]:
if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "h3_cell":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, test_df):
            df["lat"], df["lon"] = zip(*df[SPATIAL_UNIT].map(h3.cell_to_latlng))

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        # create 
        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "census_tract":
        print("Encoding: latlong and Unit: census_tract")
        # 1. Load the census tract CSV and parse the_geom (WKT) into geometry

        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(11)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check — catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "community_area":
        print("Encoding: latlong and Unit: community_area")
        # 1. Load the community area CSV and parse the_geom (WKT) into geometry
        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  # the_geom is already lon/lat degrees

        gdf["lon"] = gdf.geometry.centroid.x
        gdf["lat"] = gdf.geometry.centroid.y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(2)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check — catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]

Encoding: latlong and Unit: community_area


C:\Users\bkran\AppData\Local\Temp\ipykernel_23112\2754262089.py:50: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x
C:\Users\bkran\AppData\Local\Temp\ipykernel_23112\2754262089.py:51: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y


### Spatial Encoding: OneHotEncoding

In [73]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "community_area"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train (in case a community_area is missing)
    X_test = X_test.reindex(columns=train_columns, fill_value=0)

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
    X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)


### Spatial Encoding: Spatial Embedding

In [74]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "h3_cell"):
    # Get all unique H3 cells from your data
    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])

    # Build regions_gdf: GeoDataFrame with H3 cell polygons
    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        # h3 returns (lat, lon) tuples, shapely needs (lon, lat)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Build features_gdf: features per H3 cell (use POI counts from your data)
    # Take the mean of POI features per cell (they should be constant per cell anyway)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Build joint_gdf: maps each region to the features that fall within it
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    # Fit the embedder
    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    # embeddings is a DataFrame indexed by h3_cell with 32 columns

    # Merge embeddings into train/val/test
    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = train_df_grid[feature_cols]
elif (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "census_tract"): 
    print("help")


In [75]:
train_df.head()

,community_area,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,...,skyc1_SCT,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,trip_demand,lat,lon
0,44,5,1.224647e-16,-1.000000,0.000000,1.000000,0.000000,1.0,24.030,66.0600,...,0,0,0,30.0,1.0,24.0,6.0,2,41.740206,-87.61597
1,44,2,5.000000e-01,-0.866025,0.433884,-0.900969,0.000000,1.0,29.585,36.5875,...,0,0,0,30.0,1.0,24.0,6.0,1,41.740206,-87.61597
2,44,25,5.000000e-01,-0.866025,-0.781831,0.623490,0.866025,-0.5,13.890,89.7850,...,0,0,0,30.0,1.0,24.0,6.0,3,41.740206,-87.61597
3,44,11,5.000000e-01,-0.866025,0.433884,-0.900969,-0.866025,0.5,24.585,69.0400,...,0,0,0,30.0,1.0,24.0,6.0,3,41.740206,-87.61597
4,44,14,1.224647e-16,-1.000000,-0.974928,-0.222521,-0.866025,0.5,29.778,53.1160,...,0,0,0,30.0,1.0,24.0,6.0,3,41.740206,-87.61597


### Create y

In [76]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_train_grid = train_df_grid[TARGET_COL]

In [77]:
feature_cols

['community_area',
 'month_sin',
 'month_cos',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'tmpc',
 'relh',
 'sknt',
 'vsby',
 'p01m',
 'skyc1_BKN',
 'skyc1_CLR',
 'skyc1_FEW',
 'skyc1_OVC',
 'skyc1_SCT',
 'skyc1_VV ',
 'is_holiday',
 'food_drink',
 'landmark',
 'shop',
 'train_station']

In [78]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_grid = scaler.fit_transform(X_train_grid)

In [79]:
X_train

array([[ 0.22468373, -0.11641334, -1.53860128, ..., -0.57635541,
         0.0454235 , -0.09095812],
       [ 0.22468373,  0.59491658, -1.3477549 , ..., -0.57635541,
         0.0454235 , -0.09095812],
       [ 0.22468373,  0.59491658, -1.3477549 , ..., -0.57635541,
         0.0454235 , -0.09095812],
       ...,
       [ 0.80965114, -0.11641334,  1.31039218, ..., -0.57635541,
        -0.45972229, -0.28680214],
       [ 0.80965114,  0.59491658,  1.1195458 , ..., -0.57635541,
        -0.45972229, -0.28680214],
       [ 0.80965114,  0.59491658,  1.1195458 , ..., -0.57635541,
        -0.45972229, -0.28680214]], shape=(274961, 23))

## Grid Search

In [80]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVC(),
        param_grid=grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.45450623036829935 best params: {'C': 1, 'kernel': 'linear'}
rbf_sigmoid best score: 0.46999998499248874 best params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
poly best score: 0.4050062056059058 best params: {'C': 0.1, 'degree': 3, 'gamma': 0.1, 'kernel': 'poly'}
Overall best: rbf_sigmoid {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}


In [81]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV score: 0.46999998499248874


In [82]:
#missing_mask = test_df["lat"].isna()
#print(test_df.loc[missing_mask, SPATIAL_UNIT].unique())

In [83]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

## Training Model

In [84]:
# Train SVC 
best_model.fit(X_train, y_train)

: 

: 

## Testing Model

In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[18605   471    96  5921  2521]
 [  959 93086  1727  8517    35]
 [ 1356 30912  1159 12668    23]
 [ 6039 14662   874 24101   129]
 [ 2158     1     1    12 11219]]
              precision    recall  f1-score   support

        High       0.64      0.67      0.66     27614
         Low       0.67      0.89      0.76    104324
         Mid       0.30      0.03      0.05     46118
    Mid High       0.47      0.53      0.50     45805
   Very High       0.81      0.84      0.82     13391

    accuracy                           0.62    237252
   macro avg       0.58      0.59      0.56    237252
weighted avg       0.56      0.62      0.56    237252



## Save Model and Grid Search

In [ ]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")
dump(grid_search, "../models/grid_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_"+ SPATIAL_ENCODING + ".joblib")

['../models/test/grid_community_svc.joblib']